In [3]:
import time
from collections import deque
import numpy as np
from talib import stream

# 現在の実装
class MovingAverage:
    def __init__(self, window_size: int):
        self.window_size = window_size
        self.queue = deque()
        self.running_sum = 0.0
        self.ma = 0.0
        self.prev_ma = 0.0
    
    def update(self, value: float) -> float:
        if len(self.queue) >= self.window_size:
            self.running_sum -= self.queue.popleft()
        self.queue.append(value)
        self.running_sum += value
        self.prev_ma = self.ma
        self.ma = self.running_sum / len(self.queue)
        return self.ma

# TA-Lib版
class MovingAverageTA:
    def __init__(self, window_size: int):
        self.window_size = window_size
        self.buffer = deque(maxlen=window_size)
        self.ma = 0.0
        self.prev_ma = 0.0
    
    def update(self, value: float) -> float:
        self.buffer.append(value)
        if len(self.buffer) >= self.window_size:
            self.prev_ma = self.ma
            buffer_array = np.array(self.buffer, dtype=float)
            self.ma = stream.SMA(buffer_array, timeperiod=self.window_size)
        return self.ma

# ベンチマーク
def benchmark(ma_class, iterations=100000):
    ma = ma_class(30)
    start = time.perf_counter()
    for i in range(iterations):
        ma.update(float(i))
    elapsed = time.perf_counter() - start
    return elapsed

# 実行
time_custom = benchmark(MovingAverage)
time_talib = benchmark(MovingAverageTA)

print(f"現在の実装: {time_custom:.4f}秒")
print(f"TA-Lib版:   {time_talib:.4f}秒")
print(f"速度比:     {time_talib/time_custom:.2f}倍遅い")

現在の実装: 0.0185秒
TA-Lib版:   0.2179秒
速度比:     11.79倍遅い


In [4]:
# window_sizeを変えて測定
for window_size in [10, 30, 100, 300]:
    print(f"\n=== window_size = {window_size} ===")
    
    ma1 = MovingAverage(window_size)
    start = time.perf_counter()
    for i in range(100000):
        ma1.update(float(i))
    time1 = time.perf_counter() - start
    
    ma2 = MovingAverageTA(window_size)
    start = time.perf_counter()
    for i in range(100000):
        ma2.update(float(i))
    time2 = time.perf_counter() - start
    
    print(f"現在の実装: {time1:.4f}秒")
    print(f"TA-Lib版:   {time2:.4f}秒")
    print(f"速度比:     {time2/time1:.2f}倍")


=== window_size = 10 ===
現在の実装: 0.0213秒
TA-Lib版:   0.1863秒
速度比:     8.75倍

=== window_size = 30 ===
現在の実装: 0.0197秒
TA-Lib版:   0.2257秒
速度比:     11.44倍

=== window_size = 100 ===
現在の実装: 0.0195秒
TA-Lib版:   0.3608秒
速度比:     18.53倍

=== window_size = 300 ===
現在の実装: 0.0212秒
TA-Lib版:   0.7530秒
速度比:     35.49倍
